# Greedy edge BPE

`GreedyBPECoarsener` makes the same global pair choice as ordinary edge BPE. After choosing `(A, B)`, it immediately learns the continuation pairs `((A, B), B)`, `(((A, B), B), B)`, and so on until no newly created parent has a current `B` child.

`num_merges` counts **global seed choices**, so one seed can emit several ordinary schema-1 edge rules. The continuation rules ignore `min_pair_count`; only the initial seed pair must meet that threshold.

In [ ]:
import networkx as nx

from tree_coarsening import EdgeBPECoarsener
from tree_coarsening.coarseners.greedy_bpe import GreedyBPECoarsener

In [ ]:
def make_star(arity: int) -> nx.DiGraph:
    graph = nx.DiGraph()
    graph.add_node(0, label="A", time=0.0, uid="root")
    for node in range(1, arity + 1):
        graph.add_node(node, label="B", time=float(node), uid=f"leaf-{node}")
        graph.add_edge(0, node)
    return graph


tree = make_star(6)
tree.number_of_nodes(), tree.number_of_edges()

In [ ]:
ordinary = EdgeBPECoarsener(
    num_merges=1,
    min_pair_count=2,
    model_id="ordinary-example",
).fit([tree])

greedy = GreedyBPECoarsener(
    num_merges=1,
    min_pair_count=2,
    model_id="greedy-example",
).fit([tree])

ordinary_encoded = ordinary.transform(tree)
greedy_encoded = greedy.transform(tree)

print("ordinary emitted rules:", len(ordinary.history_))
print("ordinary output nodes: ", ordinary_encoded.number_of_nodes())
print("greedy emitted rules:  ", len(greedy.history_))
print("greedy output nodes:   ", greedy_encoded.number_of_nodes())
print("greedy raw counts:     ", [row["count"] for row in greedy.history_])

In [ ]:
def raw_signature(graph: nx.DiGraph):
    nodes = {data["uid"]: dict(data) for _, data in graph.nodes(data=True)}
    edges = frozenset(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"])
        for parent, child in graph.edges
    )
    return nodes, edges, dict(graph.graph)


restored = greedy.decode(greedy_encoded)
assert raw_signature(restored) == raw_signature(tree)
assert greedy_encoded.number_of_nodes() == 1
print("Exact round trip passed.")

## Large repeated-child runs

Each forced continuation is still one ordinary edge contraction, so a sweep of `k` children creates a left-deep exact recipe of depth `k`. The corrected shared structural implementation stores derived site/root counts privately and uses iterative equality, hashing, and representation. Normal transformation and full decoding therefore do not need a Python call stack of depth `k`.

This removes the recursion failure, but not every cost of applying `k` separate rules: fitting, transformation, and especially full decoding still do real work for every continuation. Very large stars should therefore be benchmarked rather than assumed to be constant-time.

In [ ]:
deep_tree = make_star(600)
deep_greedy = GreedyBPECoarsener(
    num_merges=1,
    min_pair_count=2,
    model_id="greedy-deep-example",
).fit([deep_tree])

deep_encoded = deep_greedy.transform(deep_tree)
deep_restored = deep_greedy.decode(deep_encoded)

assert deep_encoded.number_of_nodes() == 1
assert raw_signature(deep_restored) == raw_signature(deep_tree)
print(f"Swept and restored {deep_tree.number_of_nodes() - 1} children.")